# GraphPad Prism Data Explorer

Parse `.prism` files and plot XY data with mean ± SEM across replicates.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from prism_parser import parse_prism

PRISM_FILE = "Prism_Data.prism"
sheets = parse_prism(PRISM_FILE)
print(f"Loaded {len(sheets)} sheets: {list(sheets.keys())}")

# ---------- Color scheme configuration ----------
# Choose a matplotlib colormap for the dose columns (e.g. -12 through -6).
# Good options: "viridis", "plasma", "inferno", "magma", "cividis",
#               "coolwarm", "RdYlBu", "Spectral", "turbo"
DOSE_CMAP = "plasma"

# Fixed colors for the control / reference datasets
COLOR_BUFFER = "grey"
COLOR_FSK = "red"

# ---------- Axis limits ----------
X_MIN = 0    # minutes
X_MAX = 30   # minutes


def get_color_map(dataset_names, cmap_name=DOSE_CMAP):
    """Return a dict mapping dataset name -> color.

    Numeric-looking dose labels (e.g. '-11', '-6') are spread across
    the chosen colormap.  'buffer' and '10uM Fsk' get fixed colors.
    """
    dose_names = [n for n in dataset_names if n not in ("buffer", "10uM Fsk")]
    dose_names.sort(key=lambda s: float(s))
    cmap = mpl.colormaps[cmap_name].resampled(max(len(dose_names), 1))
    colors = {}
    for i, name in enumerate(dose_names):
        colors[name] = cmap(i / max(len(dose_names) - 1, 1))
    colors["buffer"] = COLOR_BUFFER
    colors["10uM Fsk"] = COLOR_FSK
    return colors

## Plot all sheets — mean ± SEM per dataset

In [ ]:
for sheet_title, sheet in sheets.items():
    df = sheet.df
    x = df["X"].values
    y_data = df.drop(columns="X")
    dataset_names = y_data.columns.get_level_values("dataset").unique()
    colors = get_color_map(dataset_names)

    fig, ax = plt.subplots(figsize=(10, 5))
    for ds_name in dataset_names:
        ds_cols = y_data[ds_name]
        mean = ds_cols.mean(axis=1)
        sem = ds_cols.sem(axis=1)
        ax.plot(x, mean, label=ds_name, color=colors[ds_name])
        ax.fill_between(x, mean - sem, mean + sem, alpha=0.2, color=colors[ds_name])

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_xlabel(sheet.xlabel)
    ax.set_ylabel(sheet.ylabel)
    ax.set_title(sheet.graph_title)
    ax.legend(loc="best", fontsize="small")
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    plt.tight_layout()

    filename = sheet.graph_title.replace(" ", "_").replace("/", "-") + ".png"
    fig.savefig(filename, dpi=150)
    print(f"Saved {filename}")

    plt.show()

## Inspect a single sheet

In [3]:
# Change the sheet name here to explore different datasets
SHEET = list(sheets.keys())[0]
sheet = sheets[SHEET]
df = sheet.df
print(f"Sheet: {SHEET}")
print(f"Graph title: {sheet.graph_title}")
print(f"X label: {sheet.xlabel}")
print(f"Y label: {sheet.ylabel}")
print(f"Shape: {df.shape}")
print(f"X range: {df['X'].min():.2f} – {df['X'].max():.2f}")
print(f"Datasets: {df.drop(columns='X').columns.get_level_values('dataset').unique().tolist()}")
df.head(10)

Sheet: CAMYEN-Giantin
Graph title: CAMYEN-Giantin + GLP-1R
X label: time (min)
Y label: BRET
Shape: (33, 25)
X range: 0.00 – 33.27
Datasets: ['buffer', '-11', '-10', '-9', '-8', '-7', '-6', '10uM Fsk']


/tmp/ipykernel_619235/1194864190.py:11: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  print(f"Datasets: {df.drop(columns='X').columns.get_level_values('dataset').unique().tolist()}")


dataset        X    buffer                           -11                      \
replicate                1         2         3         1         2         3   
0           0.00 -0.000111 -0.000244  0.000356  0.007189  0.001822 -0.002244   
1           1.00  0.002022 -0.003911  0.001889 -0.009278 -0.000944  0.001389   
2           2.00 -0.001911  0.004156 -0.002244  0.002089 -0.000878  0.000856   
3           4.27 -0.002411 -0.000144  0.002556  0.003989 -0.005778 -0.003644   
4           5.27 -0.004444  0.002222  0.002222 -0.002344  0.002689  0.011222   
5           6.27 -0.003744 -0.000178  0.003922 -0.002744  0.004189  0.007422   
6           7.27 -0.004011  0.001556  0.002456 -0.003411 -0.010678 -0.004844   
7           8.27 -0.008611  0.000856  0.007756 -0.021811 -0.002878 -0.004444   
8           9.27 -0.009511 -0.002444  0.011956 -0.020811  0.002122 -0.002744   
9          10.27 -0.007711  0.000456  0.007256 -0.016811 -0.004578 -0.004044   

dataset         -10                      ...        -8        -7            \
replicate         1         2         3  ...         3         1         2   
0         -0.000011 -0.002811 -0.001244  ...  0.001022  0.005222  0.005022   
1         -0.000978 -0.002078 -0.002111  ... -0.000244 -0.004544 -0.006444   
2          0.000989  0.004889  0.003356  ... -0.000778 -0.000678  0.001422   
3         -0.008811 -0.003111 -0.000844  ... -0.009578 -0.010778 -0.016578   
4         -0.017344 -0.010844 -0.004878  ... -0.014411 -0.021511 -0.013611   
5          0.000056 -0.016044 -0.004578  ... -0.012111 -0.023711 -0.019911   
6         -0.015411 -0.013711 -0.002944  ... -0.006178 -0.027278 -0.011678   
7         -0.017611 -0.014611 -0.003444  ... -0.013378 -0.032178 -0.016478   
8         -0.017911 -0.022411 -0.010644  ... -0.014278 -0.042478 -0.029778   
9         -0.018311 -0.010911 -0.012244  ... -0.013778 -0.035178 -0.026478   

dataset                    -6                      10uM Fsk            \
replicate         3         1         2         3         1         2   
0          0.004889  0.006356  0.002222  0.000289  0.001622 -0.000844   
1         -0.001078 -0.004511 -0.001344 -0.001778 -0.002544 -0.003111   
2         -0.003811 -0.001844 -0.000878  0.001489  0.000922  0.003956   
3         -0.005611 -0.013144 -0.003378 -0.003011 -0.293378 -0.296644   
4         -0.020544 -0.015278 -0.014411 -0.008144 -0.323611 -0.333978   
5         -0.006244 -0.022778 -0.003111 -0.011944 -0.326811 -0.340478   
6         -0.016011 -0.024444 -0.013378 -0.018911 -0.326678 -0.336244   
7         -0.031211 -0.034644 -0.015278 -0.021111 -0.319278 -0.334044   
8         -0.027511 -0.040144 -0.015378 -0.024511 -0.317978 -0.327044   
9         -0.026711 -0.030644 -0.019678 -0.020711 -0.302378 -0.310544   

dataset              
replicate         3  
0          0.002022  
1         -0.000544  
2         -0.001478  
3         -0.292378  
4         -0.329211  
5         -0.335811  
6         -0.325878  
7         -0.329078  
8         -0.322078  
9         -0.313278  

[10 rows x 25 columns]